# Description

In this notebook, I will run the inference through:
- The provided test case.
- My test case. 

In [1]:
import os
import math
import pandas as pd
import numpy as np
import ast 
import re
import torch
import torch.nn.functional as F
import textwrap
from typing import List
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 1. Test on single sample

In [2]:
SENTINEL = "<extra_id_0>"
PREFIX   = "predict_if_condition: "

In [3]:
def _clean(text: str) -> str:
    # Remove any leftover sentinels and tidy whitespace
    for i in range(100):
        text = text.replace(f"<extra_id_{i}>", "")
    text = re.sub(r"\s+", " ", text.strip())
    if text.endswith(":"):
        text = text[:-1].rstrip()
    return text

def load_model(model_dir: str):
    # Always load a T5 SentencePiece tokenizer
    tok = T5Tokenizer.from_pretrained(model_dir, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

    # Make sure generation IDs are set (some checkpoints lose these)
    if model.config.pad_token_id is None and tok.pad_token_id is not None:
        model.config.pad_token_id = tok.pad_token_id
    if model.config.eos_token_id is None and tok.eos_token_id is not None:
        model.config.eos_token_id = tok.eos_token_id
    if model.config.decoder_start_token_id is None:
        model.config.decoder_start_token_id = model.config.pad_token_id

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    return tok, model, device


@torch.no_grad()
def predict_condition(model, tok, device, code_snippet: str, max_new_tokens: int = 24) -> str:

    # Replace <mask> with the sentinel and add the task prefix
    code = code_snippet.replace("<mask>", SENTINEL)
    src  = PREFIX + code

    batch = tok(src, return_tensors="pt", truncation=True, max_length=1024)
    batch.pop("token_type_ids", None)
    batch = {k: v.to(device) for k, v in batch.items()}

    out = model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        num_beams=1,
        early_stopping=True,
        decoder_start_token_id=model.config.decoder_start_token_id,
        eos_token_id=model.config.eos_token_id,
        pad_token_id=model.config.pad_token_id,
        length_penalty=0.0,
    )
    return _clean(tok.decode(out[0], skip_special_tokens=True))


def post_process_condition(text: str, max_chars: int = 500) -> str:
    """
    Simple, robust cleaner for predicted conditions.
    - remove sentinels
    - cut at newline or trailing colon
    - normalize a few artifacts
    - collapse generic repetitions (words, parentheses)
    - dedupe repeated tail n-grams (e.g., `_type == "negative"` over and over)
    """
    if not isinstance(text, str):
        return ""

    s = text

    # 1) remove T5 sentinels
    s = re.sub(r"<extra_id_\d+>", "", s)

    # 2) cut at newline or trailing colon
    s = re.split(r"[\r\n]+|:\s*$", s, maxsplit=1)[0]

    # 3) normalize common artifacts / casings
    s = re.sub(r"\bnone[_\s]*type\b", "NoneType", s, flags=re.IGNORECASE)
    s = re.sub(r"\btrue\b", "True", s, flags=re.IGNORECASE)
    s = re.sub(r"\bfalse\b", "False", s, flags=re.IGNORECASE)
    s = re.sub(r"\bnone\b", "None", s, flags=re.IGNORECASE)

    # 4) collapse obvious repeats
    s = re.sub(r"(\b\w+\b)(?:\s*\1){2,}\b", r"\1", s)          # word word word -> word
    s = re.sub(r"(\([^()]*\))(?:\s*\1){2,}", r"\1", s)         # (x)(x)(x) -> (x)
    s = re.sub(r"(?:_?type){2,}\b", "type", s, flags=re.IGNORECASE)  # _type_type... -> type

    # 5) generic tail n-gram dedupe (handles `_type == "negative"` * N)
    def dedupe_tail_ngrams(t: str, max_n: int = 5) -> str:
        toks = re.findall(r'\w+|[^\s\w]', t)  # simple tokenization: words or single punct
        changed = True
        while changed:
            changed = False
            L = len(toks)
            for n in range(min(max_n, L // 2), 0, -1):
                # check if the last 2*n tokens are two identical n-grams
                if L >= 2 * n and toks[L-2*n:L-n] == toks[L-n:L]:
                    # remove one repetition
                    toks = toks[:L-n]
                    changed = True
                    break
        return "".join(
            [tok if re.fullmatch(r'[^\w\s]', tok) else (" " + tok) for tok in toks]
        ).strip()

    s = dedupe_tail_ngrams(s, max_n=5)

    # 6) collapse spaces and cap length
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_chars:
        s = s[:max_chars].rstrip()

    return s


def _mask_span_by_pos(code: str, sl: int, sc: int, el: int, ec: int, sentinel=SENTINEL) -> str:
    lines = code.splitlines(keepends=True)
    before = "".join(lines[:sl-1]) + lines[sl-1][:sc]
    after  = lines[el-1][ec:] + "".join(lines[el:])
    return before + sentinel + after

def extract_mask_all_ifs(code: str):
    """
    Yield dicts for EVERY ast.If (incl. elif/nested) with:
      - masked_code: the function with that single test replaced by SENTINEL
      - gold_cond:   exact source of the test (target)
      - idx:         0-based index of the if in source order
    """
    out = []
    code = code.strip("\n")
    if not code:
        return out
    try:
        tree = ast.parse(code)
    except Exception:
        return out

    if_nodes = [n for n in ast.walk(tree) if isinstance(n, ast.If)]
    if_nodes.sort(key=lambda n: (getattr(n, "lineno", 10**9), getattr(n, "col_offset", 10**9)))

    for i, n in enumerate(if_nodes):
        if not (hasattr(n.test, "lineno") and hasattr(n.test, "end_lineno")):
            continue
        # get exact condition text
        cond_text = None
        try:
            cond_text = ast.get_source_segment(code, n.test)
        except Exception:
            try:
                cond_text = ast.unparse(n.test)
            except Exception:
                cond_text = None
        if not cond_text:
            continue
        cond_text = cond_text.strip()
        if not cond_text:
            continue

        masked = _mask_span_by_pos(code, n.test.lineno, n.test.col_offset,
                                        n.test.end_lineno, n.test.end_col_offset,
                                        sentinel=SENTINEL)
        out.append({"idx": i, "masked_code": masked, "gold_cond": cond_text})
    return out

def _normalize(s: str) -> str:
    # light normalization for fair exact-match comparison
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\(\s*", "(", s)
    s = re.sub(r"\s*\)", ")", s)
    if s.endswith(":"):
        s = s[:-1].rstrip()
    return s



def evaluate_function(model, tok, device, func_src: str, *, max_new_tokens=64, num_beams=4):
    """
    For a single function string:
      - create one masked sample per `if`,
      - run predict_condition(...) for each,
      - compare normalized exact match,
    Returns: pandas DataFrame with one row per `if` and summary printed.
    """

    items = extract_mask_all_ifs(func_src)
    rows = []
    for it in items:
        src = f"{PREFIX}{it['masked_code']}"
        
        pred = predict_condition(model, tok, device, src)  
        pred = post_process_condition(pred)

        true_norm = _normalize(it["gold_cond"])
        pred_norm = _normalize(pred)
        
        
        rows.append({
            "if_index": it["idx"],
            "true_norm": true_norm,
            "pred_norm": pred_norm,
            "exact_match": true_norm == pred_norm
        })

    return pd.DataFrame(rows, columns=["if_index", "true_norm", "pred_norm", "exact_match"])


def complete_masked_function(
    model,
    tok,
    device,
    func_with_mask: str,
    *,
    max_new_tokens: int = 24
) -> str:
    """
    Fill a single `<mask>` inside an `if <mask>:` with the model's predicted condition
    and return the completed function string.

    - Input  : Python function source that contains exactly one `<mask>` token.
    - Output : Same function with `<mask>` replaced by the predicted condition.
    """
    if not isinstance(func_with_mask, str) or "<mask>" not in func_with_mask:
        return func_with_mask

    # Get condition text (predict_condition already converts <mask> -> SENTINEL and adds PREFIX)
    raw_pred = predict_condition(
        model, tok, device, func_with_mask, max_new_tokens=max_new_tokens
    )
    cond = post_process_condition(raw_pred)

    # Replace ONLY the first `<mask>` occurrence to avoid surprising edits
    completed = func_with_mask.replace("<mask>", cond, 1)

    # Minor tidying
    completed = re.sub(r"\s+\)", ")", re.sub(r"\(\s+", "(", completed))
    completed = re.sub(r"\s+:", ":", completed)
    return completed


def _extract_if_spans(code: str):
    """
    Return a list of dicts (sorted by source order) with:
      - idx:        0-based if order
      - sl, sc:     start line/col of condition
      - el, ec:     end line/col of condition
      - cond_text:  exact source text of the condition
    """
    out = []
    code = code.rstrip("\n")
    if not code:
        return out

    try:
        tree = ast.parse(code)
    except Exception:
        return out

    if_nodes = [n for n in ast.walk(tree) if isinstance(n, ast.If)]
    if_nodes.sort(key=lambda n: (getattr(n, "lineno", 10**9),
                                 getattr(n, "col_offset", 10**9)))

    for i, n in enumerate(if_nodes):
        if not (hasattr(n.test, "lineno") and hasattr(n.test, "end_lineno")):
            continue

        try:
            cond_text = ast.get_source_segment(code, n.test)
        except Exception:
            try:
                cond_text = ast.unparse(n.test)
            except Exception:
                cond_text = None

        if not cond_text:
            continue

        out.append({
            "idx": i,
            "sl": n.test.lineno,
            "sc": n.test.col_offset,
            "el": n.test.end_lineno,
            "ec": n.test.end_col_offset,
            "cond_text": cond_text
        })
    return out


@torch.no_grad()
def predict_condition_with_score(model, tok, device, code_snippet: str, max_new_tokens: int = 24):
    """
    Predict condition and compute average log-prob + probability score.
    """
    code = code_snippet.replace("<mask>", SENTINEL)
    src  = PREFIX + code

    batch = tok(src, return_tensors="pt", truncation=True, max_length=1024)
    batch.pop("token_type_ids", None)
    batch = {k: v.to(device) for k, v in batch.items()}

    gen = model.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        num_beams=1,
        output_scores=True,
        return_dict_in_generate=True,
        early_stopping=True,
        decoder_start_token_id=model.config.decoder_start_token_id,
        eos_token_id=model.config.eos_token_id,
        pad_token_id=model.config.pad_token_id,
    )

    pred_text = _clean(tok.decode(gen.sequences[0], skip_special_tokens=True))
    pred_text = post_process_condition(pred_text)

    # Compute average log-probability
    gen_len = len(gen.scores)
    chosen_ids = gen.sequences[0][-gen_len:]
    logprobs = []
    for step, logits in enumerate(gen.scores):
        lp = F.log_softmax(logits[0], dim=-1)
        token_id = int(chosen_ids[step].item())
        logprobs.append(float(lp[token_id].item()))

    avg_logprob = sum(logprobs) / max(1, len(logprobs))
    prob = math.exp(avg_logprob)  # normalized confidence

    return pred_text, prob


def complete_first_if(model, tok, device, func_src: str, *, max_new_tokens=32):
    """
    Detects the first `if` condition in a function, masks it,
    predicts the missing condition, and returns the completed function.
    """
    func_src = func_src.strip()
    if not func_src:
        return "Empty function input", "Empty function input", 0.0, func_src

    # Parse AST to find first if
    try:
        tree = ast.parse(func_src)
    except Exception as e:
        return "Parse code error", "Parse error", 0.0, func_src

    first_if = None
    for node in ast.walk(tree):
        if isinstance(node, ast.If):
            first_if = node
            break
    if first_if is None:
        return "NO IF", "NO IF", 0.0, func_src

    # Extract condition text
    true_if = None
    try:
        true_if = ast.get_source_segment(func_src, first_if.test)
    except Exception:
        try:
            true_if = ast.unparse(first_if.test)
        except Exception:
            true_if = None
    if not true_if:
        print("Could not extract condition text.")
        return None

    # Mask and predict
    masked = _mask_span_by_pos(func_src,
                               first_if.test.lineno, first_if.test.col_offset,
                               first_if.test.end_lineno, first_if.test.end_col_offset,
                               sentinel=SENTINEL)
    pred, score = predict_condition_with_score(model, tok, device, masked, max_new_tokens=max_new_tokens)
    pred = post_process_condition(pred)

    # Build completed version
    lines = func_src.splitlines(keepends=True)
    def _off(ls, line, col): return sum(len(x) for x in ls[:line-1]) + col
    start = _off(lines, first_if.test.lineno, first_if.test.col_offset)
    end   = _off(lines, first_if.test.end_lineno, first_if.test.end_col_offset)
    completed = func_src[:start] + pred + func_src[end:]

    return true_if, pred, score, completed

In [4]:
model_dir = "t5_if_finetuned"

tok, model, device = load_model(model_dir)

In [5]:
func = """
def check(x):
    if x > 0:
        return "positive"
    else:
        return "negative"
"""

true_if, pred, score, completed = complete_first_if(model, tok, device, func, max_new_tokens=32)

print()
print(f"Original : {true_if}")
print(f"Predicted: {pred}")
print(f"Score    : {round(score, 4)}")
print(completed)


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Original : x > 0
Predicted: x==" positive"
Score    : 0.6395
def check(x):
    if x==" positive":
        return "positive"
    else:
        return "negative"


# 2. Inference on the test case of prof Antonio

In [6]:
test_df = pd.read_csv("benchmark_if_only.csv")
print(f"Shape of test_df: {test_df.shape}")
test_df.head()

Shape of test_df: (330, 5)


,id,code,code_tokens,docstring,docstring_tokens
0,1,"def register_component_producer(self, componen...","['def', 'register_component_producer', '(', 's...",Register a component producer if the atom is r...,"['Register', 'a', 'component', 'producer', 'if..."
1,2,"def process_in_batches(tx, query, data, batch_...","['def', 'process_in_batches', '(', 'tx', ',', ...",Execute database queries in specified data bat...,"['Execute', 'database', 'queries', 'in', 'spec..."
2,3,"def filter_property(self, siid: int, piid: int...","['def', 'filter_property', '(', 'self', ',', '...",Check if a property identifier exists in a cac...,"['Check', 'if', 'a', 'property', 'identifier',..."
3,4,"def _save_ontology_to_db(self, ontology: ""Onto...","['def', '_save_ontology_to_db', '(', 'self', '...",Save ontology data to a database if not alread...,"['Save', 'ontology', 'data', 'to', 'a', 'datab..."
4,5,def _build_merge_function(self):\n \n\n...,"['def', '_build_merge_function', '(', 'self', ...",Create a function to add trace timestamps to c...,"['Create', 'a', 'function', 'to', 'add', 'trac..."


In [7]:
out_df = []

for i, row in test_df.iterrows():
    func_src = row["code"]
    true_if, pred, score, completed = complete_first_if(model, tok, device, func_src, max_new_tokens=32)
    score = round(score, 4) * 100.0 
    
    out_df.append({
        "function_source": func_src,
        "completed_function": completed,
        "true_condition": true_if,
        "predicted_condition": pred,
        "confidence_score": score
    })
    
out_df = pd.DataFrame(out_df, columns=[
    "function_source",
    "completed_function",
    "true_condition",
    "predicted_condition",
    "confidence_score"
])
print(f"Output DataFrame shape: {out_df.shape}")
out_df.head()

Output DataFrame shape: (330, 5)


,function_source,completed_function,true_condition,predicted_condition,confidence_score
0,"def register_component_producer(self, componen...","def register_component_producer(self, componen...",atom_name in self.atoms,self. _component_producers is not None_id_id_i...,81.15
1,"def process_in_batches(tx, query, data, batch_...","def process_in_batches(tx, query, data, batch_...",NO IF,NO IF,0.00
2,"def filter_property(self, siid: int, piid: int...","def filter_property(self, siid: int, piid: int...",self._cache and 'properties' in self._cache an...,siid== piid or siid== piid or siid== piid or s...,55.73
3,"def _save_ontology_to_db(self, ontology: ""Onto...","def _save_ontology_to_db(self, ontology: ""Onto...",self.ontology_table_name in self.falkordb.list...,self. name not in ontology_storage_graph_stora...,52.35
4,def _build_merge_function(self):\n \n\n...,def _build_merge_function(self):\n \n\n...,do_trace_tagging,do_trace_tagging not None_trace_tagging not No...,77.56


In [8]:
out_df.to_csv("provided-testset.csv", index=False)

# 2. My test case

In [9]:
# Load training data for evaluation
PATH_FILE_DATA = os.path.join(os.getcwd(), "dataset", "processed", "processed_data.csv")

df = pd.read_csv(PATH_FILE_DATA)
list_python_function = df["method_code"].tolist()
list_python_function = list_python_function[:1_000] # limit to first 1000 for speed
print(f"Number of functions to evaluate: {len(list_python_function)}")

Number of functions to evaluate: 1000


In [10]:
my_out_df = []

for func_src in list_python_function:
    true_if, pred, score, completed = complete_first_if(model, tok, device, func_src, max_new_tokens=32)
    my_out_df.append({
        "function_source": func_src,
        "completed_function": completed,
        "true_condition": true_if,
        "predicted_condition": pred,
        "confidence_score": round(score, 4) * 100.0
    })
    
my_out_df = pd.DataFrame(my_out_df, columns=[
    "function_source",
    "completed_function",
    "true_condition",
    "predicted_condition",
    "confidence_score"
])
print(f"My Output DataFrame shape: {my_out_df.shape}")
my_out_df.head()

My Output DataFrame shape: (1000, 5)


,function_source,completed_function,true_condition,predicted_condition,confidence_score
0,"def custom_round(x, decimal_places=2):\n st...","def custom_round(x, decimal_places=2):\n st...","leading_zeros >= 1 and before_decimal == ""0""",after_decimal==' 0' in after_decimal,74.66
1,def scito_decimal(sci_str):\n def split_exp...,def scito_decimal(sci_str):\n def split_exp...,"""."" in decimal_str",len( coefficient)== 2 len( coefficient),71.80
2,"def normalize(res, round_to=2):\n # we ...","def normalize(res, round_to=2):\n # we ...","""."" in res","isinstance( res, str) str=""."" in res str)",37.00
3,def add_(args):\n\n return normalize(sum(ar...,def add_(args):\n\n return normalize(sum(ar...,NO IF,NO IF,0.00
4,def subtract_(args):\n\n res = args[0]\n ...,def subtract_(args):\n\n res = args[0]\n ...,NO IF,NO IF,0.00


In [11]:
my_out_df.to_csv("generated-testset.csv", index=False)